# Getting Started with Computer Use

Claude can interact with computer environments through the **Computer Use tool** — a beta feature that provides screenshot capture, mouse control, and keyboard input for autonomous desktop interaction.

Unlike regular tool use where you define custom input schemas, the Computer Use tool has a schema built directly into Claude's model. You simply declare the tool with your display dimensions, and Claude responds with precise actions like `left_click` at coordinates, `type` text strings, or `key` presses.

**How it works at a high level:**

1. You send Claude a user message along with the computer use tool definition
2. Claude decides to use the tool and returns an action (e.g., click at `[640, 400]`)
3. Your application executes that action in a sandboxed environment and captures a screenshot
4. You send the screenshot back to Claude as a `tool_result`
5. Claude analyzes the result and decides the next action — this cycle is the **agent loop**

> **Important:** Computer use is a client-side tool. Claude doesn't directly connect to any desktop. Your application is responsible for executing actions, capturing screenshots, and returning results. For security, always run this in an isolated sandbox (Docker container or VM).

In this cookbook, you'll learn:
- How to define the computer use tool and configure your API client
- The full set of available actions (mouse, keyboard, screenshot, zoom)
- How to build the agent loop that drives the interaction
- How to handle screenshots and coordinate scaling
- How to augment computer use with companion tools (bash, text editor)
- Security best practices and known limitations

**Prerequisites:**
- An Anthropic API key
- Python 3.11+
- `anthropic` Python SDK

## 1. Setup

Install the Anthropic SDK and initialize the client. Computer use is a beta feature, so we pass the beta flag via the `betas` parameter on each API call (not as a default header).

In [ ]:
%pip install anthropic

In [ ]:
import anthropic

client = anthropic.Anthropic()  # Uses ANTHROPIC_API_KEY env variable

# We'll use a current model that supports the latest computer use beta.
# claude-sonnet-4-6 supports computer-use-2025-11-24 with computer_20251124 tool type.
MODEL_NAME = "claude-sonnet-4-6"

# The beta flag to include with each API call
COMPUTER_USE_BETA = "computer-use-2025-11-24"

## 2. Defining the Computer Use Tool

The computer use tool is a **schema-less** tool — you don't provide an `input_schema` like you would with custom tools. The schema is built into Claude's model. You just declare:

| Parameter | Description |
|-----------|-------------|
| `type` | `"computer_20251124"` (latest version) |
| `name` | `"computer"` (required, must be exactly this) |
| `display_width_px` | Width of your virtual display in pixels |
| `display_height_px` | Height of your virtual display in pixels |
| `display_number` | Optional X11 display number (e.g., `1` for `:1`) |
| `enable_zoom` | Optional, set `true` to enable the `zoom` action for inspecting screen regions at full resolution |

In [ ]:
# Define the display dimensions matching your sandbox environment.
DISPLAY_WIDTH = 1024
DISPLAY_HEIGHT = 768

# The computer use tool definition
computer_tool = {
    "type": "computer_20251124",
    "name": "computer",
    "display_width_px": DISPLAY_WIDTH,
    "display_height_px": DISPLAY_HEIGHT,
    "display_number": 1,
    # Enable zoom to let Claude inspect regions at full resolution
    "enable_zoom": True,
}

tools = [computer_tool]
print(f"Tool defined: {computer_tool['type']} at {DISPLAY_WIDTH}x{DISPLAY_HEIGHT}")

## 3. Available Actions Reference

When Claude uses the computer tool, it returns an `action` field in the tool input. Here's the complete set of actions available with `computer_20251124`:

### Mouse Actions

| Action | Parameters | Description |
|--------|-----------|-------------|
| `mouse_move` | `coordinate: [x, y]` | Move cursor to coordinates |
| `left_click` | `coordinate: [x, y]` | Left-click at coordinates |
| `right_click` | `coordinate: [x, y]` | Right-click at coordinates |
| `middle_click` | `coordinate: [x, y]` | Middle-click at coordinates |
| `double_click` | `coordinate: [x, y]` | Double-click at coordinates |
| `triple_click` | `coordinate: [x, y]` | Triple-click at coordinates |
| `left_click_drag` | `start_coordinate: [x, y]`, `coordinate: [x, y]` | Click and drag between coordinates |
| `left_mouse_down` | `coordinate: [x, y]` | Press and hold left button |
| `left_mouse_up` | `coordinate: [x, y]` | Release left button |

### Keyboard Actions

| Action | Parameters | Description |
|--------|-----------|-------------|
| `type` | `text: "string"` | Type a text string |
| `key` | `text: "key_combo"` | Press a key or combo (e.g., `"Return"`, `"ctrl+c"`) |
| `hold_key` | `text: "key"`, `duration: seconds` | Hold a key for a duration |

### Screen & Control Actions

| Action | Parameters | Description |
|--------|-----------|-------------|
| `screenshot` | *(none)* | Capture the current display |
| `scroll` | `coordinate: [x, y]`, `direction`, `amount` | Scroll in any direction |
| `wait` | `duration: seconds` | Pause between actions |
| `zoom` | `region: [x1, y1, x2, y2]` | View a screen region at full resolution (requires `enable_zoom: true`) |

## 4. Making Your First API Call

Let's send a simple request to Claude and see what kind of tool use action it returns. In a real setup, Claude would be responding to a screenshot of an actual desktop — but even without one, we can see the API contract in action.

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Take a screenshot of the current desktop so we can see what's on screen.",
    }
]

response = client.beta.messages.create(
    model=MODEL_NAME,
    max_tokens=1024,
    tools=tools,
    messages=messages,
    betas=[COMPUTER_USE_BETA],
)

print(f"Stop reason: {response.stop_reason}")
print()

for block in response.content:
    if block.type == "text":
        print(f"Text: {block.text}")
    elif block.type == "tool_use":
        print(f"Tool: {block.name}")
        print(f"Tool Use ID: {block.id}")
        print(f"Action: {block.input}")

Claude should respond with `stop_reason: "tool_use"` and a `tool_use` content block requesting a `screenshot` action. The `tool_use_id` is important — you'll need it when returning the result.

### Returning a Tool Result

After your application executes the action (e.g., captures a screenshot), you return the result as a `tool_result` message. For screenshots, you include the image as a base64-encoded content block:

In [ ]:
# In a real application, you would capture an actual screenshot here.
# This is the structure you'd use to return it to Claude:

example_tool_result = {
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": "toolu_example_id",  # Must match the tool_use_id from Claude's response
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": "<base64-encoded-screenshot-data>",
                    },
                }
            ],
        }
    ],
}

print("Tool result structure (for reference):")
import json

print(json.dumps(example_tool_result, indent=2))

## 5. Building the Agent Loop

The core of computer use is the **agent loop** — a cycle where:
1. Claude requests a tool action
2. Your application executes it in the sandbox
3. Your application returns the result (typically a screenshot)
4. Claude analyzes the result and decides the next action

This continues until Claude responds without requesting any tools (task complete) or you hit a maximum iteration limit.

Below is a complete, production-ready agent loop implementation:

In [ ]:
def execute_computer_action(action_type, params):
    """
    Execute a computer use action in your sandbox environment.

    In a real implementation, this would interface with your Docker container
    or VM to perform mouse/keyboard actions and capture screenshots.

    Returns: a list of content blocks to send back as tool_result.
    """
    if action_type == "screenshot":
        # Capture a screenshot from your sandbox
        # screenshot_bytes = capture_screenshot_from_sandbox()
        # screenshot_b64 = base64.b64encode(screenshot_bytes).decode("utf-8")
        # return [{"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": screenshot_b64}}]

        # Placeholder for demonstration
        print("  → Would capture screenshot")
        return [{"type": "text", "text": "Screenshot captured (placeholder)"}]

    elif action_type == "left_click":
        x, y = params["coordinate"]
        print(f"  → Would click at ({x}, {y})")
        return [{"type": "text", "text": f"Clicked at ({x}, {y})"}]

    elif action_type == "type":
        text = params["text"]
        print(f"  → Would type: {text!r}")
        return [{"type": "text", "text": f"Typed: {text}"}]

    elif action_type == "key":
        key = params["text"]
        print(f"  → Would press key: {key}")
        return [{"type": "text", "text": f"Pressed key: {key}"}]

    elif action_type == "mouse_move":
        x, y = params["coordinate"]
        print(f"  → Would move mouse to ({x}, {y})")
        return [{"type": "text", "text": f"Mouse moved to ({x}, {y})"}]

    elif action_type == "scroll":
        direction = params.get("direction", "down")
        amount = params.get("amount", 3)
        coord = params.get("coordinate", [0, 0])
        print(f"  → Would scroll {direction} by {amount} at ({coord[0]}, {coord[1]})")
        return [{"type": "text", "text": f"Scrolled {direction}"}]

    else:
        print(f"  → Would execute action: {action_type} with params: {params}")
        return [{"type": "text", "text": f"Executed {action_type}"}]


def agent_loop(
    user_message,
    model=MODEL_NAME,
    max_tokens=4096,
    max_iterations=10,
    system_prompt=None,
):
    """
    Run the computer use agent loop.

    Args:
        user_message: The task for Claude to accomplish.
        model: The Claude model to use.
        max_tokens: Maximum tokens per response.
        max_iterations: Safety limit to prevent runaway API costs.
        system_prompt: Optional system instructions.

    Returns:
        The full conversation messages list.
    """
    messages = [{"role": "user", "content": user_message}]

    for iteration in range(1, max_iterations + 1):
        print(f"\n--- Iteration {iteration} ---")

        # Build the API call kwargs
        api_kwargs = {
            "model": model,
            "max_tokens": max_tokens,
            "tools": tools,
            "messages": messages,
            "betas": [COMPUTER_USE_BETA],
        }
        if system_prompt:
            api_kwargs["system"] = system_prompt

        # Call the API
        response = client.beta.messages.create(**api_kwargs)

        # Add Claude's response to conversation history
        messages.append({"role": "assistant", "content": response.content})

        # Process any tool use requests
        tool_results = []
        for block in response.content:
            if block.type == "text":
                print(f"Claude says: {block.text}")
            elif block.type == "tool_use":
                action = block.input.get("action", "unknown")
                print(f"Tool request: {block.name} → action={action}")

                # Execute the action
                result_content = execute_computer_action(action, block.input)

                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result_content,
                    }
                )

        # If no tools were used, Claude is done
        if not tool_results:
            print("\n✅ Task complete — no more tool use requested.")
            return messages

        # Send tool results back and continue the loop
        messages.append({"role": "user", "content": tool_results})

    print(f"\n⚠️ Reached max iterations ({max_iterations}). Stopping.")
    return messages

Let's test the agent loop with a simple task. Since we're using placeholder actions (no real sandbox), Claude will receive text-based results instead of screenshots. In a production setup, you'd return actual screenshot images after each action.

> **Note:** Running this cell requires a valid `ANTHROPIC_API_KEY` and will make API calls.

In [ ]:
# Run the agent loop with a simple task.
# In a real setup, Claude would see actual screenshots after each action.
result = agent_loop(
    user_message="Take a screenshot to see what's currently on screen.",
    max_iterations=3,
)

## 6. Handling Screenshots

In a production implementation, returning real screenshots is crucial — it's how Claude "sees" the environment. Here's how you'd capture and return a screenshot using the `mss` library:

```python
# pip install mss Pillow
import mss
import base64
from io import BytesIO
from PIL import Image

def capture_screenshot():
    """Capture the screen and return as base64-encoded PNG."""
    with mss.mss() as sct:
        # Capture the primary monitor
        monitor = sct.monitors[1]
        screenshot = sct.grab(monitor)
        
        # Convert to PIL Image, then to base64
        img = Image.frombytes("RGB", screenshot.size, screenshot.bgra, "raw", "BGRX")
        buffer = BytesIO()
        img.save(buffer, format="PNG")
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

# Return as tool_result
screenshot_b64 = capture_screenshot()
tool_result_content = [
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": screenshot_b64,
        },
    }
]
```

For the sandboxed Docker approach, you would instead use `subprocess` to call `xdotool` or a similar tool inside the container, and read the screenshot file from disk.

## 7. Coordinate Scaling for Higher Resolutions

The API constrains images to a maximum of **1568 pixels** on the longest edge and approximately **1.15 megapixels** total. If your display exceeds these limits, you need to:

1. Resize screenshots **before** sending to Claude
2. Scale Claude's returned coordinates **back up** before executing actions

> **Exception:** Claude Opus 4.7 supports up to 2576 pixels on the long edge, and its coordinates are 1:1 with image pixels — no scale-factor conversion is needed.

In [ ]:
import math


def get_scale_factor(width, height):
    """Calculate scale factor to meet API image constraints.

    Returns a factor <= 1.0 that, when applied to the image dimensions,
    ensures the image fits within the API's limits.
    """
    long_edge = max(width, height)
    total_pixels = width * height

    long_edge_scale = 1568 / long_edge
    total_pixels_scale = math.sqrt(1_150_000 / total_pixels)

    return min(1.0, long_edge_scale, total_pixels_scale)


# Example: a 1920x1080 display
screen_w, screen_h = 1920, 1080
scale = get_scale_factor(screen_w, screen_h)
scaled_w = int(screen_w * scale)
scaled_h = int(screen_h * scale)

print(f"Original: {screen_w}x{screen_h}")
print(f"Scale factor: {scale:.4f}")
print(f"Scaled for API: {scaled_w}x{scaled_h}")
print()


def scale_coordinates_back(x, y, scale_factor):
    """Convert Claude's coordinates back to original screen space."""
    return int(x / scale_factor), int(y / scale_factor)


# If Claude says click at (400, 300) in the scaled image:
claude_x, claude_y = 400, 300
real_x, real_y = scale_coordinates_back(claude_x, claude_y, scale)
print(f"Claude coordinates: ({claude_x}, {claude_y})")
print(f"Actual screen coordinates: ({real_x}, {real_y})")

## 8. Augmenting with Companion Tools

Computer use becomes much more powerful when combined with the **bash tool** and **text editor tool**. You can include all three in the same `tools` array — only the computer use tool requires the beta header.

This lets Claude:
- Use `computer` for visual/GUI interactions
- Use `bash` for running shell commands directly
- Use `text_editor` (str_replace_based_edit_tool) for precise file editing

In [ ]:
# Full tool configuration with all three tools
full_tools = [
    {
        "type": "computer_20251124",
        "name": "computer",
        "display_width_px": 1024,
        "display_height_px": 768,
        "display_number": 1,
    },
    {
        "type": "text_editor_20250728",
        "name": "str_replace_based_edit_tool",
    },
    {
        "type": "bash_20250124",
        "name": "bash",
    },
]

print("Tools configured:")
for tool in full_tools:
    print(f"  • {tool['type']} → {tool['name']}")

In [ ]:
# Example API call with all three tools
response = client.beta.messages.create(
    model=MODEL_NAME,
    max_tokens=1024,
    tools=full_tools,
    messages=[
        {
            "role": "user",
            "content": "List the files in the current directory using bash.",
        }
    ],
    betas=[COMPUTER_USE_BETA],
)

for block in response.content:
    if block.type == "tool_use":
        print(f"Tool: {block.name}")
        print(f"Input: {block.input}")

Notice that Claude intelligently chose the `bash` tool instead of the `computer` tool for a file-listing task — no GUI interaction needed.

## 9. Security Best Practices

Computer use has unique security risks compared to standard API features. Follow these guidelines:

### Isolation
- **Always** use a dedicated virtual machine or Docker container with minimal privileges
- Never run computer use on your primary workstation
- Limit internet access to an allowlist of domains

### Data Protection
- Never expose sensitive credentials, login sessions, or personal data to the sandbox
- Use dedicated test accounts if Claude needs to log into services
- Pass credentials via `<robot_credentials>` XML tags in the prompt if needed

### Human-in-the-Loop
- Require human confirmation for consequential actions (financial transactions, accepting ToS, etc.)
- Anthropic's built-in classifiers will flag potential prompt injection in screenshots and auto-prompt for user confirmation

### Prompt Injection Defense
- Claude may follow instructions found in screenshots (e.g., text on a webpage) even if they conflict with your instructions
- Review the [prompt injection mitigation guide](https://docs.anthropic.com/docs/en/test-and-evaluate/strengthen-guardrails/mitigate-jailbreaks) for defense strategies

### Prompting Tips
- Give Claude explicit, step-by-step instructions for complex tasks
- Add a checkpoint prompt: *"After each step, take a screenshot and carefully evaluate if you have achieved the right outcome."*
- Suggest keyboard shortcuts when GUI elements (dropdowns, scrollbars) are tricky to manipulate

## 10. Known Limitations

The Computer Use feature is in beta. Be aware of these limitations:

| Limitation | Details |
|-----------|----------|
| **Latency** | Current latency may be too slow for real-time human-AI interactions. Best for background tasks (automated testing, data gathering). |
| **Vision accuracy** | Claude may make mistakes when identifying coordinates. It can hallucinate UI element positions. |
| **Tool selection** | Claude may choose unexpected tools or approaches, especially with niche applications or multi-app workflows. |
| **Spreadsheet interaction** | Cell selection can be tricky — use fine-grained controls (`left_mouse_down`, `left_mouse_up`) with modifier keys for better reliability. |
| **Account creation** | Claude's ability to create accounts on social platforms or impersonate humans is intentionally limited. |
| **Prompt injection** | Instructions found on web pages or in images can override user instructions. Always sandbox. |

## 11. Pricing Notes

Computer use follows standard [tool use pricing](https://docs.anthropic.com/docs/en/agents-and-tools/tool-use/overview#pricing) with some extras:

- **System prompt overhead:** The computer use beta adds ~466–499 tokens to the system prompt
- **Screenshot costs:** Each screenshot image adds tokens per [Vision pricing](https://docs.anthropic.com/docs/en/build-with-claude/vision)
- **Tool result tokens:** Text returned in `tool_result` blocks also counts toward input tokens
- **Companion tools:** Bash and text editor tools have their own token costs

Since the agent loop can run for many iterations, always set a `max_iterations` limit to control costs.

## 12. Next Steps

Now that you understand the Computer Use API contract, here's where to go next:

- 🐳 **[Reference Implementation](https://github.com/anthropics/anthropic-quickstarts/tree/main/computer-use-demo)** — A complete Docker-based setup with web interface, tool implementations, and agent loop. This is the fastest way to see computer use in action end-to-end.

- 📖 **[Computer Use Documentation](https://docs.anthropic.com/en/docs/build-with-claude/computer-use)** — Full API reference, including detailed tool parameters and action schemas.

- 🔧 **[Bash Tool](https://docs.anthropic.com/docs/en/agents-and-tools/tool-use/bash-tool)** / **[Text Editor Tool](https://docs.anthropic.com/docs/en/agents-and-tools/tool-use/text-editor-tool)** — Companion tools that pair well with computer use.

- 🧠 **[Extended Thinking](https://docs.anthropic.com/docs/en/build-with-claude/extended-thinking)** — Combine computer use with extended thinking for transparent step-by-step reasoning.

- 🛡️ **[Prompt Injection Mitigation](https://docs.anthropic.com/docs/en/test-and-evaluate/strengthen-guardrails/mitigate-jailbreaks)** — Essential reading for production deployments.

- 🏗️ **[Effective Harnesses for Long-Running Agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents)** — Best practices for agents that span multiple sessions.